# Unsloth Fine-tuning DeepSeek R1 Distilled Llama 8B

在这个notbook中，将演示如何使用Unsloth在医学数据集上微调DeepSeek-R1-Distill-Llama-8B模型。

1、在内蒙B区（要与数据盘在同一个大区）租用实例，将模型文件、llamafactory解压到autodl-tmp目录下，将数据文件复制到autodl-tmp目录下
这里模型文件、数据文件已经上传好，下载地址为：
- 模型（huggingface）：https://huggingface.co/unsloth/DeepSeek-R1-Distill-Llama-8B/tree/main
- 数据（huggingface）：https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/tree/main

## 按照https://acnfgkjh8azx.feishu.cn/wiki/FSbQwyvzniSK9nkfEKZcgzNFnce?fromScene=spaceOverview的方法下载模型和数据集

## 环境准备

In [1]:
# ============================================================
# 固定版本环境安装（CUDA 12.8 + PyTorch 2.10.0）
# 运行一次即可，安装完成后建议重启 Kernel
# ============================================================

# 1. 安装 PyTorch（CUDA 12.8 专用 wheel）
!pip install "torch==2.10.0" torchvision torchaudio \\
    --index-url https://download.pytorch.org/whl/cu128 -q

# 2. 安装 xformers（与 torch 2.10.0 + CUDA 12.8 匹配）
!pip install "xformers==0.0.35" \\
    --index-url https://download.pytorch.org/whl/cu128 -q

# 3. 安装 unsloth 核心（固定版本，避免 API 变动引起冲突）
!pip install "unsloth==2026.3.17" -q
!pip install "unsloth_zoo==2026.3.6" -q

# 4. 安装量化依赖
!pip install "bitsandbytes==0.49.2" -q

# 5. 安装训练框架（版本需与 unsloth 匹配）
!pip install "transformers==4.57.3" -q
!pip install "peft==0.18.1" -q
!pip install "accelerate==1.13.0" -q
!pip install "trl==0.24.0" -q

# 6. 安装数据集库
!pip install "datasets==4.3.0" -q

# 验证关键库版本
import importlib
pkgs = ["torch", "unsloth", "trl", "transformers", "bitsandbytes", "peft", "accelerate", "datasets"]
for pkg in pkgs:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg}: {m.__version__}")
    except Exception as e:
        print(f"{pkg}: 导入失败 -> {e}")

## 模型加载

**模型选择方法**
- 选择一个与您的使用案例相符的模型
- 评估您的存储、计算能力和数据集
- 选择模型和参数
- 选择基础模型还是指令模型

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

## 训练前模型推理

In [37]:
prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Please answer the following medical question.

### Question:
{}

### Response:
<think>{}"""

In [38]:
question = "一个患有急性阑尾炎的病人已经发病5天，腹痛稍有减轻但仍然发热，在体检时发现右下腹有压痛的包块，此时应如何处理？"


FastLanguageModel.for_inference(model)
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print(response[0].split("### Response:")[1])


<think>
好的，我现在需要帮助处理一个急性阑尾炎患者的情况。患者已经发病5天，腹痛稍有减轻但仍然发热。在体检时，医生发现了右下腹有压痛的包块。接下来，我应该如何处理呢？

首先，急性阑尾炎通常表现为右下腹痛，尤其是发热的情况。患者已经有5天的病程，虽然腹痛有所缓解，但仍有发热，这可能意味着炎症仍在继续，或者可能存在其他并发症。

发现右下腹有压痛的包块，这可能是阑尾的炎症性改变，或者可能是其他如腺体炎症。需要进一步明确。

接下来，可能需要进行超声检查，确定包块的性质。如若发现包块，可能是阑尾炎症，或者是其他结构如肝、脾等的病变。

如果包块是阑尾的，可能需要考虑是否需要抗生素治疗，尤其是如果患者有发热和腹痛，可能有感染风险。

如果包块是其他结构，如肝脏或脾的炎症，可能需要针对性治疗。

此外，患者的症状虽然有所缓解，但仍有发热，可能需要评估是否有感染并考虑使用抗生素，尤其是如果有不明确的来源。

同时，考虑到患者已经有5天的病程，可能需要评估是否有并发症，如感染性休克、多器官功能障碍等。

在处理过程中，可能需要进行血常规、电解质检查、肝肾功能测试等，以评估患者的全身状况。

如果包块在超声检查后确认为阑尾炎症，可能需要进一步的内窥镜检查，确认阑尾的炎症程度和是否需要切除。

如果有必要，可能需要进行抗生素治疗，同时控制患者的疼痛和发热。

此外，考虑到患者的腹痛有所缓解，可能需要调整治疗方案，避免过度使用止痛药，以防止药物过度使用带来的问题。

总结来说，处理步骤可能包括：

1. 进一步的体格检查，确认包块的位置和性质。
2. 进行超声检查，确定包块的来源。
3. 基于超声结果，决定是否需要抗生素治疗。
4. 评估患者的全身状况，进行必要的生理检查。
5. 根据检查结果，决定是否需要进一步的内窥镜检查或其他治疗手段。
6. 控制患者的疼痛和发热，防止并发症。

在处理过程中，需要保持密切观察患者的症状，及时调整治疗方案，确保患者病情得到有效控制。
</think>

根据患者的症状和检查结果，以下是处理步骤：

1. **进一步体格检查**：确认包块的位置和性质，是否存在其他异常。

2. **超声检查**：确定包块的来源，可能包括阑尾炎症或其他结构如肝、脾的炎症。

3. **抗生素治疗评估**：根据发热情况和包块性质，决定是否需要抗生素，尤其是如

## 数据集准备

医疗数据集： https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/
华佗o1的训练数据集

In [39]:
train_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Please answer the following medical question.

### Question:
{}

### Response:
<think>
{}
</think>
{}"""

In [40]:
EOS_TOKEN = tokenizer.eos_token  # 一定要添加结束标记，否则会无限生成


def formatting_prompts_func(examples):
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return {
        "text": texts,
    }

In [ ]:
from datasets import load_dataset
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", 'zh', split = "train[0:500]", trust_remote_code=True)
print(dataset.column_names)

In [42]:
dataset = dataset.map(formatting_prompts_func, batched = True)
dataset["text"][0]

'Below is an instruction that describes a task, paired with an input that provides further context.\nWrite a response that appropriately completes the request.\nBefore answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.\n\n### Instruction:\nYou are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.\nPlease answer the following medical question.\n\n### Question:\n根据描述，一个1岁的孩子在夏季头皮出现多处小结节，长期不愈合，且现在疮大如梅，溃破流脓，口不收敛，头皮下有空洞，患处皮肤增厚。这种病症在中医中诊断为什么病？\n\n### Response:\n<think>\n这个小孩子在夏天头皮上长了些小结节，一直都没好，后来变成了脓包，流了好多脓。想想夏天那么热，可能和湿热有关。才一岁的小孩，免疫力本来就不强，夏天的湿热没准就侵袭了身体。\n\n用中医的角度来看，出现小结节、再加上长期不愈合，这些症状让我想到了头疮。小孩子最容易得这些皮肤病，主要因为湿热在体表郁结。\n\n但再看看，头皮下还有空洞，这可能不止是简单的头疮。看起来病情挺严重的，也许是脓肿没治好。这样的情况中医中有时候叫做禿疮或者湿疮，也可能是另一种情况。\n\n等一下，头皮上的空洞和皮肤增厚更像是疾病已经深入到头皮下，这是不是说明有可能是流注或瘰疬？这些名字常描述头部或颈部的严重感染，特别是有化脓不愈合，又形成通道或空洞的情况。\n\n仔细想想，我怎么感觉这些症状更贴近瘰疬的表现？尤其考虑到孩子的年纪和夏天发生的季节性因素，湿热可能是主因，但可能也有火毒或者痰湿造成的滞留。\n\n回到基本

## 模型训练

使用Huggingface TRL's SFTTrainer

In [43]:
FastLanguageModel.for_training(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSN

In [44]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False, # Can make training 5x faster for short sequences.
        per_device_train_batch_size = 2,#每个设备训练批次大小
        gradient_accumulation_steps = 4,#梯度积累步数
        warmup_steps = 5,#预热步数
        max_steps = 60,#最大训练步数
        # num_train_epochs = 1, # For longer training runs!
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",#优化器
        weight_decay = 0.01,
        lr_scheduler_type = "linear",#学习率调度器类型
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

**report_to 参数用于指定训练过程中报告的目的地，可以设置为以下几种值：**

"none"：不报告任何信息。这通常用于不需要进行外部监控的本地训练。

"wandb"：将训练日志、模型等信息发送到 Weights & Biases (wandb)，一个用于机器学习实验管理和可视化的平台。如果你希望在训练过程中查看图表、损失函数变化等信息，并跟踪模型的训练过程，选择 wandb。

"tensorboard"：将训练日志发送到 TensorBoard，以便在本地或远程访问并可视化训练过程。可以通过启动 tensorboard 服务器来查看训练日志。

"comet"：将训练日志发送到 Comet，这是一个类似于 Weights & Biases 的机器学习实验跟踪平台。

"mlflow"：将训练日志发送到 MLflow，一个开源的平台，用于管理机器学习生命周期。

"neptune"：将训练日志发送到 Neptune，这是另一个机器学习实验管理平台。

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 500 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 60
 "-____-"     Number of trainable parameters = 41,943,040


Step,Training Loss
1,1.515100
2,1.590900
3,1.469900
4,1.420700
5,1.444000
6,1.331000
7,1.600500
8,1.494500
9,1.629800
10,1.544500


## 微调后的评估

In [49]:
print(question)

一个患有急性阑尾炎的病人已经发病5天，腹痛稍有减轻但仍然发热，在体检时发现右下腹有压痛的包块，此时应如何处理？


In [50]:
FastLanguageModel.for_inference(model)  # Unsloth has 2x faster inference!
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print(response[0].split("### Response:")[1])


<think>
这个病人已经有五天的急性阑尾炎了，虽然他的腹痛已经稍微减轻了，但他仍然发烧，还在右下腹部有一个压痛的包块。嗯，这种情况下我觉得最重要的是要控制他的发烧。

首先，我想到的就是使用退烧药。退烧药可以很快地将他的体温降下来，这样可以缓解他的不适，减轻他的疼痛感。

然后，我想到抗生素。因为急性阑尾炎通常是由于细菌感染引起的，抗生素可以帮助清除这些细菌，减少感染的风险，保护他的阑尾。

接下来，我想到肠道保护剂。虽然他现在可能不需要立即进行手术，但肠道保护剂可以帮助他的肠道黏膜恢复，预防感染扩散，确保他的肠道健康。

最后，我想到如果包块已经被诊断为炎症性包块，手术就成了一个重要的选择。手术不仅能彻底清除病变，还能防止未来的感染。

所以，综合考虑，我觉得在这种情况下，应该首先使用退烧药来缓解发烧，然后用抗生素清除感染，接着用肠道保护剂保护肠道，最后根据情况进行手术处理。这样可以全面地保护他的健康。
</think>
对于一个患有急性阑尾炎且腹痛稍有减轻但仍然发热的病人，在右下腹部有压痛的包块时，应采取以下步骤进行处理：

1. **退烧药治疗**：首先应使用退烧药以迅速降低病人的发烧，这可以缓解其不适和疼痛感。

2. **抗生素治疗**：由于急性阑尾炎通常由细菌感染引起，抗生素可以帮助清除感染，降低感染风险，并保护阑尾。

3. **肠道保护剂**：虽然病人可能不需要立即手术，但肠道保护剂可以帮助恢复肠道黏膜，预防感染扩散，维护肠道健康。

4. **手术处理**：如果包块已被诊断为炎症性包块，应考虑进行手术以彻底清除病变，防止未来的感染。

综上所述，首先应使用退烧药缓解发烧，接着使用抗生素清除感染，使用肠道保护剂保护肠道，最后根据病情进行手术处理。<｜end▁of▁sentence｜>
